# Bronze Layer: Wikimedia Pageview Ingestion

This notebook creates the Bronze layer for the Global Attention Analytics project.

The Bronze layer is the raw ingestion layer of the Lakehouse pipeline. Hourly Wikimedia pageview files are downloaded, decompressed, lightly structured, and stored as a Delta table.

## What This Notebook Does

This notebook:

- downloads hourly Wikimedia pageview files
- reads compressed `.gz` files
- keeps English Wikipedia desktop and mobile projects
- extracts the four raw fields from each line
- adds source metadata such as date, hour, and source URL
- writes the result to a Bronze Delta table

## Input Data

The raw data comes from Wikimedia hourly pageview files.

Each raw line contains four fields:

- project
- page title
- views
- bytes sent

The project currently focuses on:

- `en` = English Wikipedia desktop
- `en.m` = English Wikipedia mobile

## Output Table

`workspace.wiki_bronze.pageviews_raw`

## Why This Step Matters

The Bronze layer keeps the data close to the original source while storing it in a structured Delta table.

This makes the pipeline reproducible and allows later notebooks to use the same raw data without downloading it again.

The Bronze table is intentionally not fully cleaned. Cleaning, normalization, and topic categorization are handled in the Silver layer.

## Current Scope

Each run loads the latest twelve completed Wikimedia hourly files after a three-hour safety delay.

The table is refreshed as a rolling twelve-hour snapshot. This keeps the project efficient in Databricks Free Edition while still supporting regularly updated analytics.

In [0]:
import urllib.request
import gzip
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark.sql("USE CATALOG workspace")

BRONZE_TABLE = "wiki_bronze.pageviews_raw"

target_projects = ["en", "en.m"]

schema = StructType([
    StructField("project", StringType(), True),
    StructField("page_title", StringType(), True),
    StructField("views", IntegerType(), True),
    StructField("bytes_sent", IntegerType(), True),
    StructField("source_date", StringType(), True),
    StructField("source_hour", IntegerType(), True),
    StructField("source_url", StringType(), True)
])

print("Using catalog: workspace")
print("Bronze table:", BRONZE_TABLE)

Using catalog: workspace
Bronze table: wiki_bronze.pageviews_raw


In [0]:
def load_pageview_hour(source_date, source_hour):
    """
    Downloads one hourly Wikimedia pageview file,
    filters English Wikipedia desktop/mobile rows,
    and returns a Spark DataFrame.
    """
    
    date_compact = source_date.replace("-", "")
    year_month = source_date[:7]
    year = source_date[:4]
    
    source_url = (
        f"https://dumps.wikimedia.org/other/pageviews/"
        f"{year}/{year_month}/pageviews-{date_compact}-{source_hour:02d}0000.gz"
    )
    
    print(f"Loading hour {source_hour:02d}: {source_url}")
    
    rows = []
    
    with urllib.request.urlopen(source_url, timeout=180) as response:
        with gzip.GzipFile(fileobj=response) as gz:
            for raw_line in gz:
                line = raw_line.decode("utf-8", errors="replace").strip()
                parts = line.split(" ")
                
                if len(parts) != 4:
                    continue
                
                project, page_title, views, bytes_sent = parts
                
                if project not in target_projects:
                    continue
                
                rows.append((
                    project,
                    page_title,
                    int(views),
                    int(bytes_sent),
                    source_date,
                    source_hour,
                    source_url
                ))
    
    print(f"Rows loaded for hour {source_hour:02d}: {len(rows)}")
    
    return spark.createDataFrame(rows, schema)

In [0]:
from datetime import datetime, timedelta, timezone

# Dynamic 12-hour ingestion
# Loads the last 12 completed Wikimedia hourly files with a safety delay.

source_timezone = timezone.utc
safety_delay_hours = 3
hours_back = 12

now_utc = datetime.now(source_timezone)

# Use a safety delay because the newest Wikimedia hourly files may not be available immediately.
end_time = now_utc.replace(minute=0, second=0, microsecond=0) - timedelta(hours=safety_delay_hours)

hours_to_load = [
    end_time - timedelta(hours=i)
    for i in range(hours_back)
]

# Sort oldest to newest, so the table is written in natural time order.
hours_to_load = sorted(hours_to_load)

print("Loading dynamic 12-hour window:")
for dt in hours_to_load:
    print(dt.strftime("%Y-%m-%d %H:00 UTC"))

for i, dt in enumerate(hours_to_load):
    source_date = dt.strftime("%Y-%m-%d")
    source_hour = dt.hour
    
    hour_df = load_pageview_hour(source_date, source_hour)
    
    write_mode = "overwrite" if i == 0 else "append"
    
    hour_df.write \
        .format("delta") \
        .mode(write_mode) \
        .option("overwriteSchema", "true") \
        .saveAsTable(BRONZE_TABLE)
    
    print(f"Written {source_date} hour {source_hour:02d} to {BRONZE_TABLE} with mode: {write_mode}")

print("Dynamic 12-hour Bronze ingestion finished.")

Loading dynamic 12-hour window:
2026-06-14 18:00 UTC
2026-06-14 19:00 UTC
2026-06-14 20:00 UTC
2026-06-14 21:00 UTC
2026-06-14 22:00 UTC
2026-06-14 23:00 UTC
2026-06-15 00:00 UTC
2026-06-15 01:00 UTC
2026-06-15 02:00 UTC
2026-06-15 03:00 UTC
2026-06-15 04:00 UTC
2026-06-15 05:00 UTC
Loading hour 18: https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-180000.gz
Rows loaded for hour 18: 2383508
Written 2026-06-14 hour 18 to wiki_bronze.pageviews_raw with mode: overwrite
Loading hour 19: https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-190000.gz
Rows loaded for hour 19: 2424455
Written 2026-06-14 hour 19 to wiki_bronze.pageviews_raw with mode: append
Loading hour 20: https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-200000.gz
Rows loaded for hour 20: 2426824
Written 2026-06-14 hour 20 to wiki_bronze.pageviews_raw with mode: append
Loading hour 21: https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews

In [0]:
spark.sql("""
SELECT
  COUNT(*) AS rows,
  SUM(views) AS total_views,
  COUNT(DISTINCT CONCAT(source_date, '-', source_hour)) AS loaded_hour_slots,
  MIN(source_date) AS min_date,
  MAX(source_date) AS max_date,
  MIN(source_hour) AS min_hour,
  MAX(source_hour) AS max_hour
FROM wiki_bronze.pageviews_raw
""").show()

+--------+-----------+-----------------+----------+----------+--------+--------+
|    rows|total_views|loaded_hour_slots|  min_date|  max_date|min_hour|max_hour|
+--------+-----------+-----------------+----------+----------+--------+--------+
|26842334|  133670753|               12|2026-06-14|2026-06-15|       0|      23|
+--------+-----------+-----------------+----------+----------+--------+--------+

